# PromptMaxxing - Towards Autonomous Prompt Iteration

### [@tastengyal](https://x.com/tastengyal) - [Rezi](https://x.com/rezi_ai)

## Motivation

- The prompt is a moat most AI application companies today.
- We all strive to make our AI inferences provide the best responses. In other words, the surrounding prompts are critical.
- However, maintaining and improving prompts iteratively while keeping it model agnostic is extremely slow and often un-scientific.
- If not done right, it results in ["Prompt Debt"](https://www.dbreunig.com/2026/06/22/the-problem-is-prompt-debt.html)

# A Solution That Works For Us

### [DSPy](https://dspy.ai) - Program, don’t prompt your LLMs

- DSPy lets you declare a prompt as a typed **signature** and treat improving it as an *optimization problem* against an evaluation metric you trust. 
- Super fast iteration instead of hand-tweaking wording.
- With a clear evaluator, it turns prompt engineering into a measurable, repeatable loop.  

For any improvement (modify or add a new capability), you simply add/modify a criterion to the **evaluator**, not hand-edited instructions into your System Prompt.

# Example - Spell Backward

We'll walk through the loop end to end on a small, objectively checkable task:  

                spelling a word backward

Because there's exactly one right answer, the evaluator is plain exact matching — no LLM judge needed.

## The dataset

`spell_back.csv` holds `instruction, question, answer` rows — each `question` is a word and `answer` is that word reversed (e.g. `requiz → ziuqer`). The examples are drawn from the [Open Thoughts](https://www.open-thoughts.ai/) open reasoning dataset.

In [15]:
import dspy
import pandas as pd
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()

load_dotenv()

# The LM we optimize.
lm = dspy.LM("openai/gpt-4o-mini", temperature=1, max_tokens=2048)
dspy.configure(lm=lm)
dspy.disable_logging()

df = pd.read_csv("./spell_back.csv")
console.print(Panel.fit(
    f"[bold green]Loaded dataset[/bold green] · {len(df):,} rows",
    border_style="green",
))
df.head()

╭──────────────────────────╮
│ Loaded dataset · 40 rows │
╰──────────────────────────╯

,instruction,question,answer
0,Spell this word backward (example: sun -> nus):,requiz,ziuqer
1,Spell this word backward (example: sun -> nus):,getup,puteg
2,Spell this word backward (example: sun -> nus):,palpation,noitaplap
3,Spell this word backward (example: sun -> nus):,Karamojo,ojomaraK
4,Spell this word backward (example: sun -> nus):,cicala,alacic


## Step 1 — Baseline with DSPy

In [DSPy](https://dspy.ai) you declare a task as a typed **signature** — a spec of inputs and outputs whose docstring instruction becomes the actual prompt. This reframes prompt engineering as an *optimization problem*.

Our baseline is the *unoptimized* program: given a word, it returns the reversed word. That docstring instruction is exactly what the optimizer will later try to rewrite and beat.

In [16]:
class SpellBackward(dspy.Signature):
    """Spell the given word backward (example: sun -> nus)."""

    word: str = dspy.InputField(desc="The word to reverse.")
    reversed_word: str = dspy.OutputField(desc="The word spelled backward.")


baseline = dspy.Predict(SpellBackward)


def row_to_example(row) -> dspy.Example:
    return dspy.Example(word=row["question"], reversed_word=row["answer"]).with_inputs("word")


# 60/20/20 split with a seeded shuffle.
shuffled = df.sample(frac=1.0, random_state=0).reset_index(drop=True)
n = len(shuffled)
n_train, n_val = int(n * 0.6), int(n * 0.2)
trainset = [row_to_example(shuffled.iloc[i]) for i in range(n_train)]
valset = [row_to_example(shuffled.iloc[i]) for i in range(n_train, n_train + n_val)]
testset = [row_to_example(shuffled.iloc[i]) for i in range(n_train + n_val, n)]

split_table = Table(title="Dataset split", box=None, pad_edge=False)
split_table.add_column("Split", style="bold cyan")
split_table.add_column("Examples", justify="right", style="white")
split_table.add_row("Train", f"{len(trainset):,}")
split_table.add_row("Validation", f"{len(valset):,}")
split_table.add_row("Test", f"{len(testset):,}")
console.print(split_table)

   Dataset split    
Split       Examples
Train             24
Validation         8
Test               8

## Step 2 — The evaluator: exact match

To optimize anything, we need a metric to optimize towards. Because the task has exactly one correct answer, the evaluator is trivial and fixed: a prediction passes iff it equals the reversed word exactly (whitespace-trimmed).

We run the baseline over the held-out test split to get the number the optimizer has to beat.

In [17]:
from dspy.evaluate import Evaluate


def exact_match(gold, pred, trace=None) -> bool:
    got = (getattr(pred, "reversed_word", "") or "").strip()
    return got == gold.reversed_word.strip()


evaluator = Evaluate(devset=testset, metric=exact_match, num_threads=4, display_progress=True)
baseline_result = evaluator(baseline)
console.print(Panel(
    f"[bold]Baseline test accuracy[/bold]  [yellow]{baseline_result.score:.1f}%[/yellow]",
    border_style="yellow",
))

Average Metric: 2.00 / 8 (25.0%): 100%|██████████| 8/8 [00:00<00:00, 1246.08it/s]


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Baseline test accuracy  25.0%                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 3 - Optimizing with GEPA

[GEPA](https://dspy.ai/api/optimizers/GEPA/overview/) (Genetic-Pareto) is DSPy's reflective prompt optimizer. It evolves the prompt's *instruction* in a loop:

1. **Run** the current prompt on a batch of training examples.
2. **Reflect** — a strong reflection model reads the metric's *textual feedback* (here, the correct answer for each miss) and proposes a rewritten instruction that targets those specific failures.
3. **Select** — candidates are scored on the validation split, and GEPA keeps a *Pareto frontier* of the best performers rather than a single winner, so it doesn't overfit to one kind of example.

In [18]:
def gepa_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    got = (getattr(pred, "reversed_word", "") or "").strip()
    expected = gold.reversed_word.strip()
    if got == expected:
        return dspy.Prediction(score=1.0, feedback="Correct.")
    feedback = f"Wrong. For '{gold.word}' the correct answer is '{expected}', but got '{got}'."
    return dspy.Prediction(score=0.0, feedback=feedback)


reflection_lm = dspy.LM("openai/gpt-5.4-mini", temperature=1.0, max_tokens=4096)

optimizer = dspy.GEPA(
    metric=gepa_metric,
    auto="light",
    reflection_lm=reflection_lm,
    reflection_minibatch_size=3,
    num_threads=4,
    track_stats=False, # True if you wanna see the logs
    seed=0,
)

optimized = optimizer.compile(baseline, trainset=trainset, valset=valset)

GEPA Optimization:   0%|          | 0/412 [00:00<?, ?rollouts/s]

Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:00<00:00, 5599.87it/s]

GEPA Optimization:  35%|███▌      | 146/412 [00:00<00:00, 1403.71rollouts/s]


Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:00<00:00, 4651.72it/s]

GEPA Optimization:  74%|███████▍  | 306/412 [00:00<00:00, 1487.07rollouts/s]


Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:00<00:00, 4425.93it/s]


GEPA Optimization: 100%|█████████▉| 411/412 [00:00<00:00, 1368.20rollouts/s]


## Step 4 - Inspect the evolved instruction

GEPA optimizes the signature's *instruction* — compare it before vs. after.

In [19]:
console.print(Panel(
    baseline.signature.instructions,
    title="[bold yellow]Baseline instruction[/bold yellow]",
    border_style="yellow",
))
for name, predictor in optimized.named_predictors():
    console.print(Panel(
        predictor.signature.instructions,
        title=f"[bold green]Optimized instruction · {name}[/bold green]",
        border_style="green",
    ))

╭───────────────────────────────────────────── Baseline instruction ──────────────────────────────────────────────╮
│ Spell the given word backward (example: sun -> nus).                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Optimized instruction · self ──────────────────────────────────────────╮
│ You will be given a single input field:                                                                         │
│                                                                                                                 │
│ - word: a string containing a word or name                                                                      │
│                                                                                                                 │
│ Task:                                                                                                           │
│ - Reverse the exact sequence of characters in the given word.                                                   │
│ - Output only the reversed word, with no extra words, punctuation, spaces, or explanation.                      │
│                                                                                                                 │
│ Important:                                                                                                      │
│ - Preserve every character exactly as it appears in the input, including uppercase/lowercase letters.           │
│ - Do not add, remove, or rearrange any characters except by reversing their order.                              │
│ - Do not insert spaces or split the word.                                                                       │
│ - Do not normalize spelling or otherwise modify the input.                                                      │
│ - The reversal must be exact and character-by-character; for example, if the input is "Hedera", the correct     │
│ output is "aredeH" (not "Aredah" or any other altered form).                                                    │
│                                                                                                                 │
│ How to solve:                                                                                                   │
│ - Read the word from left to right.                                                                             │
│ - Write it back from right to left, character by character.                                                     │
│                                                                                                                 │
│ Examples:                                                                                                       │
│ - sun -> nus                                                                                                    │
│ - impoverish -> hsirevopmi                                                                                      │
│ - Chachapuya -> ayupahcahC                                                                                      │
│ - thrawcrook -> koorcwarht                                                                                      │
│ - calculous -> suoluclac                                                                                        │
│ - cicala -> alacic                                                                                              │
│ - roquist -> tsiuqor                                                                                            │
│ - Hedera -> aredeH                                                                                              │
│ - turbofan -> nafobrut                                                                                          │
│                                                                                                                 │
│ Note:                                                                                                           │
│ - The word may be a common word, a technical term, a proper name, or a specialized vocabulary item. Reverse the │
│ entire string exactly as given, regardless of familiar

## Step 5 — Compare baseline vs optimized

Score both programs on the held-out test set (unseen during optimization), show every example side by side, and persist the winner.

In [20]:
def predict_all(program, examples):
    return [(getattr(program(word=ex.word), "reversed_word", "") or "").strip() for ex in examples]


expected = [ex.reversed_word.strip() for ex in testset]
base_out = predict_all(baseline, testset)
opt_out = predict_all(optimized, testset)

# Per-example side-by-side, with a ✅/❌ marker on each prediction.
compare = pd.DataFrame({
    "word": [ex.word for ex in testset],
    "expected": expected,
    "baseline": [f"{'✅' if b == e else '❌'} {b}" for b, e in zip(base_out, expected)],
    "optimized": [f"{'✅' if o == e else '❌'} {o}" for o, e in zip(opt_out, expected)],
})

n = len(testset)
base_hits = sum(b == e for b, e in zip(base_out, expected))
opt_hits = sum(o == e for o, e in zip(opt_out, expected))
base_acc, opt_acc = base_hits / n, opt_hits / n

results = Table(title=f"Results · held-out test set ({n} examples)", box=None)
results.add_column("Program", style="bold")
results.add_column("Accuracy", justify="right")
results.add_column("Correct", justify="right")
results.add_row("Baseline", f"{base_acc:.1%}", f"{base_hits}/{n}")
results.add_row("Optimized", f"[green]{opt_acc:.1%}[/green]", f"{opt_hits}/{n}")
results.add_row("Improvement", f"[bold]{opt_acc - base_acc:+.1%}[/bold]", "—")
console.print(results)

optimized.save("spell_back_gepa.json")
console.print("[bold green]✓[/bold green] Saved optimized program to [cyan]spell_back_gepa.json[/cyan]")

compare

 Results · held-out test set (8 
           examples)            
 Program      Accuracy  Correct 
 Baseline        25.0%      2/8 
 Optimized       37.5%      3/8 
 Improvement    +12.5%        —

✓ Saved optimized program to spell_back_gepa.json

,word,expected,baseline,optimized
0,serigraphy,yhpargires,❌ yhgarise res,❌ yhparigres
1,polyphage,egahpylop,✅ egahpylop,❌ egahpyloP
2,trisected,detcesirt,❌ detcisert,❌ detcisertet
3,fearlessly,ylsselraef,❌ ysselraef,❌ ylsraefle
4,convener,renevnoc,❌ revennoc,✅ renevnoc
5,jongleur,ruelgnoj,❌ regluonj,✅ ruelgnoj
6,Karamojo,ojomaraK,❌ ojomarak,❌ ojomarak
7,requiz,ziuqer,✅ ziuqer,✅ ziuqer


### The post-optimization hillclimb

<div align="center">

<table>
<tr><th>Baseline</th><th>Optimized</th><th>Gain</th></tr>
<tr><td><strong>25.0%</strong><br><sub>2 / 8 correct</sub></td><td><strong>37.5%</strong><br><sub>3 / 8 correct</sub></td><td><strong>+12.5 pp</strong><br><sub>one more example</sub></td></tr>
</table>

</div>

On the held-out test set, GEPA's rewritten instruction moves the exact-match score upward by one successful prediction. That is a genuine, measurable hillclimb—but a modest one: **5 of 8 examples still fail**, and the small test set means this result is evidence of progress. At scale, the effects are tremendous!

# Takeaways

We turned prompt engineering into a measurable, repeatable loop:

1. **Declare the task** as a DSPy signature — the docstring *is* the prompt.
2. **Define what "good" means** with a fixed evaluator. Here it's exact match; for open-ended tasks you'd swap in an LLM judge, but the loop is identical.
3. **Optimize automatically** — GEPA rewrites the instruction from the metric's written feedback, no hand-editing.

The mindset shift: when you have a new capability to add to an existing system prompt, don't hand-edit the prompt — add it to the eval criteria and let the LLMs optimize the prompt for you via optimizers like GEPA.

<div align="center"><img src="fig1.png" alt="Prompt optimization workflow" width="40%"></div>

# Putting human experts in the loop

The most valuable part is the evaluator/judge because it is what drives the loop. And that's where real field experience pays off: real lawyers, recruiters, clinicians — whoever owns the domain are better positioned to design and calibrate the evaluation criteria so the optimizer is chasing genuine quality, not a proxy that a developer assumes. They can also enrich the datasets by seeding or grading examples that make the synthetic dataset sharper. 

In a nutshell, bringing the net disribution of the llm inference system closer to real-world distribution still requires human experts.

### [Rezi](www.rezi.ai) is positioning itself to offer access to 4+ million experts across 100+ industries to help companies build and maintain state of the art AI Applications.

Reach out to [@tastengyal](https://x.com/tastengyal) if you find this interesting!

## References & further reading

- **DSPy** — signatures and optimizers: [dspy.ai](https://dspy.ai)
- **GEPA** — the optimizer behind Step 3. Agrawal et al., *"GEPA: Reflective Prompt Evolution Can Outperform Reinforcement Learning"* (2025), [arXiv:2507.19457](https://arxiv.org/abs/2507.19457)
- **Open Thoughts** — the open reasoning dataset the examples come from: [open-thoughts.ai](https://www.open-thoughts.ai/)